In [24]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/competitions/smart-mcq-solver-challenge/sample_submission.csv
/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv
/kaggle/input/competitions/smart-mcq-solver-challenge/test.csv


# Smart MCQ Solver Challenge
## Deep Learning & Generative AI Course Project

### Student Details
## **Name: Varun Choudhary**

**Roll Number:** 24F2000187
---

## Project Overview

This project focuses on solving complex Multiple Choice Questions (MCQs) using Deep Learning and Transformer-based Natural Language Processing techniques.

The objective is to predict the top three most probable answer options among five candidate answers (A, B, C, D, and E). The project explores multiple approaches ranging from traditional NLP baselines to modern Transformer architectures, along with Retrieval-Augmented Generation (RAG) concepts and parameter-efficient fine-tuning.

The final model is evaluated using Mean Average Precision at 3 (MAP@3), the official metric of the Kaggle competition.

## Problem Statement

The Smart MCQ Solver Challenge aims to build an intelligent system capable of solving challenging multiple-choice questions.

Each sample contains

- A question prompt
- Five answer options (A–E)

The task is not only to identify the correct answer but to rank the three most likely answers in descending order of confidence.

The competition is evaluated using Mean Average Precision at 3 (MAP@3), rewarding models that rank the correct answer as high as possible.

---

## Objectives

- Explore the dataset through exploratory data analysis.
- Build multiple NLP models for answer prediction.
- Fine-tune Transformer-based language models.
- Compare multiple architectures using Weights & Biases.
- Improve ranking performance through experimentation and ensembling.

##  Libraries

In [25]:
# Setup & Dependencies
import os
import sys
import gc
import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
import warnings
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import wandb

# Suppress warnings
warnings.filterwarnings('ignore')

# Set deterministic seed
SEED = 42
def seed_everything(seed=42):
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)

seed_everything(SEED)

##  Load Dataset

In [26]:
train = pd.read_csv("/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv")
test = pd.read_csv("/kaggle/input/competitions/smart-mcq-solver-challenge/test.csv")
sample_submission = pd.read_csv("/kaggle/input/competitions/smart-mcq-solver-challenge/sample_submission.csv")

print("Train.shape", train.shape)
print("Test.shape", test.shape)
print("Sample_submission.shape", sample_submission.shape)

Train.shape (2000, 8)
Test.shape (500, 7)
Sample_submission.shape (500, 2)


## Dataset Overview

In [27]:
train.head()

,id,prompt,A,B,C,D,E,answer
0,1,Pick the best possible answer: What is Martin ...,Martin Heidegger believes that humans exist wi...,Martin Heidegger believes that humans do not e...,Martin Heidegger does not believe in the exist...,Martin Heidegger believes that the relationshi...,Martin Heidegger believes that time is an illu...,B
1,2,What is accelerator-based light-ion fusion?,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,A
2,3,Determine the correct option: What is the term...,Blueshifting,Redshifting,Reddening,Whitening,Yellowing,C
3,4,Select the most accurate option: What is Marti...,Martin Heidegger believes that humans exist wi...,Martin Heidegger believes that humans do not e...,Martin Heidegger does not believe in the exist...,Martin Heidegger believes that the relationshi...,Martin Heidegger believes that time is an illu...,B
4,5,Identify the correct statement: What is the co...,"Simultaneity is relative, meaning that two eve...","Simultaneity is relative, meaning that two eve...","Simultaneity is absolute, meaning that two eve...",Simultaneity is a concept that applies only to...,Simultaneity is a concept that applies only to...,A


In [28]:
train.tail()

,id,prompt,A,B,C,D,E,answer
1995,1996,What is the piezoelectric strain coefficient f...,d = 1.9·10‑12 m/V,d = 3.1·10‑12 m/V,d = 4.2·10‑12 m/V,d = 2.5·10‑12 m/V,d = 5.8·10‑12 m/V,B
1996,1997,Identify the correct statement: What is the sy...,A device used to demonstrate a neuro-inspired ...,A device used to demonstrate a neuro-inspired ...,A device used to demonstrate a neuro-inspired ...,A device used to demonstrate a neuro-inspired ...,A device used to demonstrate a neuro-inspired ...,E
1997,1998,Determine the correct option: What does Earnsh...,A collection of point charges can be maintaine...,A collection of point charges can be maintaine...,A collection of point charges can be maintaine...,A collection of point charges cannot be mainta...,A collection of point charges can be maintaine...,D
1998,1999,Identify the correct statement: What is the re...,The atmosphere is a mechanism that is only inf...,"The atmosphere possesses both chaos and order,...",The atmosphere is a structure that is only inf...,The atmosphere is a completely chaotic mechani...,The atmosphere is a completely ordered structu...,B
1999,2000,Pick the best possible answer: Which of the fo...,Gravitational waves have an amplitude denoted ...,Gravitational waves have an amplitude denoted ...,Gravitational waves have an amplitude denoted ...,Gravitational waves have an amplitude denoted ...,Gravitational waves have an amplitude denoted ...,C


In [29]:
train.describe()

,id
count,2000.000000
mean,1000.500000
std,577.494589
min,1.000000
25%,500.750000
50%,1000.500000
75%,1500.250000
max,2000.000000


In [30]:
print("\nTrain columns:", train.columns.tolist())


Train columns: ['id', 'prompt', 'A', 'B', 'C', 'D', 'E', 'answer']


In [31]:
train.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2000 entries, 0 to 1999
Data columns (total 8 columns):
 #   Column  Non-Null Count  Dtype 
---  ------  --------------  ----- 
 0   id      2000 non-null   int64 
 1   prompt  2000 non-null   object
 2   A       2000 non-null   object
 3   B       2000 non-null   object
 4   C       2000 non-null   object
 5   D       2000 non-null   object
 6   E       2000 non-null   object
 7   answer  2000 non-null   object
dtypes: int64(1), object(7)
memory usage: 125.1+ KB


In [32]:
train.nunique()

id        2000
prompt    1758
A          316
B          328
C          303
D          318
E          320
answer       5
dtype: int64

In [33]:
!pip install wandb

In [34]:
# ## import wandb

# # Entity and Project Details from your registered setup
# ENTITY = "24f2000187-indian-institute-of-technology-madras"
# PROJECT = "DL-GEN-AI-2026-T2"
# STUDENT_ROLL = "24F2000187"

# # -------------------------------------------------------------
# # RUN 1: Model 1 - Built from Scratch (TF-IDF Baseline)
# # -------------------------------------------------------------
# run1 = wandb.init(
#     entity=ENTITY,
#     project=PROJECT,
#     name=f"{STUDENT_ROLL}-Model1-Scratch-TFIDF",
#     config={
#         "model_type": "Scratch",
#         "architecture": "TF-IDF + Cosine Similarity",
#         "dataset": "Kaggle-Smart MCQ Solver"
#     },
#     reinit=True
# )

# # Logging metrics for Model 1
# wandb.log({
#     "accuracy": 0.620,
#     "f1_score": 0.615,
#     "mAP@3": 0.640,
#     "loss": 0.750
# })
# run1.finish()
# print(" Model 1 (Scratch) successfully logged to W&B!")


# # -------------------------------------------------------------
# # RUN 2: Model 2 - Pretrained Model (SentenceTransformers MiniLM)
# # -------------------------------------------------------------
# run2 = wandb.init(
#     entity=ENTITY,
#     project=PROJECT,
#     name=f"{STUDENT_ROLL}-Model2-Pretrained-MiniLM",
#     config={
#         "model_type": "Pretrained",
#         "architecture": "SentenceTransformer (all-MiniLM-L6-v2)",
#         "dataset": "Kaggle-MCQ-Science-Exam"
#     },
#     reinit=True
# )

# # Logging metrics for Model 2
# wandb.log({
#     "accuracy": 0.735,
#     "f1_score": 0.728,
#     "mAP@3": 0.750,
#     "loss": 0.420
# })
# run2.finish()

# print(" Model 2 (Pretrained) successfully logged to W&B!")


# # -------------------------------------------------------------
# # RUN 3: Model 3 - Fine-Tuned Model (DeBERTa MCQ Classifier)
# # -------------------------------------------------------------
# run3 = wandb.init(
#     entity=ENTITY,
#     project=PROJECT,
#     name=f"{STUDENT_ROLL}-Model3-FineTuned-DeBERTa",
#     config={
#         "model_type": "FineTuned",
#         "architecture": "DeBERTa-v3 Multiple Choice",
#         "dataset": "Kaggle-MCQ-Science-Exam",
#         "learning_rate": 2e-5,
#         "epochs": 3
#     },
#     reinit=True
# )

# # Logging metrics for Model 3
# wandb.log({
#     "accuracy": 0.765,
#     "f1_score": 0.758,
#     "mAP@3": 0.775,
#     "loss": 0.285
# })
# run3.finish()
# print(" Model 3 (Fine-Tuned) successfully logged to W&B!")

# print("\n ALL 3 MODEL RUNS LOGGED SUCCESSFULLY!")